# GeoPulse — Day 3
## Feature Engineering

### Objective

Create analytical features from the mobility dataset that will support
temporal analysis, mobility classification, and future spatial analytics.

### Features

- Time period classification
- Peak hour indicator
- Weekend indicator
- Month
- Week of month
- Movement type code

In [7]:
import pandas as pd
import numpy as np

mobility_df = pd.read_csv("mobility_pings.csv")
stores_df = pd.read_csv("stores.csv")
zones_df = pd.read_csv("zones.csv")

mobility_df["timestamp"] = pd.to_datetime(mobility_df["timestamp"])

print("Datasets loaded successfully.")
print("Mobility shape:", mobility_df.shape)

#time period 
def classify_time_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

mobility_df["time_period"] = mobility_df["hour"].apply(classify_time_period)

print(mobility_df["time_period"].value_counts())

#peak hour flag 
hourly_counts = mobility_df["hour"].value_counts()

peak_hour = hourly_counts.idxmax()

mobility_df["is_peak_hour"] = (
    mobility_df["hour"] == peak_hour
).astype(int)

print("Peak hour:", f"{peak_hour:02d}:00")
print("\nPeak hour flag:")
print(mobility_df["is_peak_hour"].value_counts())

#weekend flag
mobility_df["is_weekend"] = (
    mobility_df["day_of_week"]
    .isin(["Saturday", "Sunday"])
    .astype(int)
)

print(mobility_df["is_weekend"].value_counts())

#month
mobility_df["month"] = mobility_df["timestamp"].dt.month

print("Month distribution:")
print(mobility_df["month"].value_counts().sort_index())

#week of month
mobility_df["week_of_month"] = (
    ((mobility_df["timestamp"].dt.day - 1) // 7) + 1
)

print("Week of month distribution:")
print(mobility_df["week_of_month"].value_counts().sort_index())

#movement type code
movement_mapping = {
    "Walking": 1,
    "Driving": 2,
    "Public Transport": 3
}

mobility_df["movement_type_code"] = (
    mobility_df["movement_type"].map(movement_mapping)
)

print(mobility_df[
    ["movement_type", "movement_type_code"]
].drop_duplicates())

#check new dataset
print("Updated columns:")
print(mobility_df.columns.tolist())

print("\nFirst 5 rows:")
print(
    mobility_df[
        [
            "ping_id",
            "timestamp",
            "hour",
            "time_period",
            "is_peak_hour",
            "is_weekend",
            "month",
            "week_of_month",
            "movement_type",
            "movement_type_code"
        ]
    ].head()
)

#feature validation
print("Missing values in new features:")
print(
    mobility_df[
        [
            "time_period",
            "is_peak_hour",
            "is_weekend",
            "month",
            "week_of_month",
            "movement_type_code"
        ]
    ].isnull().sum()
)

#save engineered dataset
mobility_df.to_csv(
    "mobility_pings_featured.csv",
    index=False
)

print("Feature-engineered dataset saved successfully.")

Datasets loaded successfully.
Mobility shape: (5000, 10)
time_period
Night        1662
Morning      1462
Afternoon    1055
Evening       821
Name: count, dtype: int64
Peak hour: 02:00

Peak hour flag:
is_peak_hour
0    4774
1     226
Name: count, dtype: int64
is_weekend
0    3685
1    1315
Name: count, dtype: int64
Month distribution:
month
1    5000
Name: count, dtype: int64
Week of month distribution:
week_of_month
1    1174
2    1159
3    1166
4    1153
5     348
Name: count, dtype: int64
       movement_type  movement_type_code
0            Walking                   1
1            Driving                   2
16  Public Transport                   3
Updated columns:
['ping_id', 'anonymous_user_id', 'timestamp', 'latitude', 'longitude', 'device_type', 'movement_type', 'date', 'hour', 'day_of_week', 'time_period', 'is_peak_hour', 'is_weekend', 'month', 'week_of_month', 'movement_type_code']

First 5 rows:
  ping_id           timestamp  hour time_period  is_peak_hour  is_weekend  \
0  

# Day 3 Conclusion

Feature engineering was performed on the GeoPulse synthetic mobility dataset.

### Features Created

- `time_period` — classified mobility records into Morning, Afternoon, Evening, and Night
- `is_peak_hour` — identifies records occurring during the highest-mobility hour
- `is_weekend` — identifies Saturday and Sunday records
- `month` — extracts the month from the timestamp
- `week_of_month` — identifies the week within each month
- `movement_type_code` — converts movement categories into numerical codes

### Validation

The newly created features were checked for missing values and data consistency.

The feature-engineered dataset was saved separately as:

`mobility_pings_featured.csv`

The original raw mobility dataset was preserved without modification.

### Day 3 Status

Completed.

The engineered features will be used in the upcoming data-processing and
spatial analytics stages of GeoPulse.